In [ ]:
import csv
import datetime as date
import json
import os
import time
import statistics

import numpy as np
import pandas as pd
import requests

from requests.exceptions import SSLError

In [ ]:
os.mkdir(path = "../data")
os.mkdir(path = '../data/download')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def reset_index(download_path, index_filename):
    """Reset index in file to 0."""
    rel_path = os.path.join(download_path, index_filename)

    with open(rel_path, 'w') as f:
        print(0, file=f)


def get_index(download_path, index_filename):
    """Retrieve index from file, returning 0 if file not found."""
    try:
        rel_path = os.path.join(download_path, index_filename)

        with open(rel_path, 'r') as f:
            index = int(f.readline())

    except FileNotFoundError:
        index = 0

    return index


def prepare_data_file(download_path, filename, index, columns):
    """Create file and write headers if index is 0."""
    if index == 0:
        rel_path = os.path.join(download_path, filename)

        with open(rel_path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=columns)
            writer.writeheader()

In [ ]:
def get_app_data(start, stop, parser, pause):
    """Return list of app data generated from parser.

    parser : function to handle request
    """
    app_data = []

    # iterate through each row of app_list, confined by start and stop
    for index, row in app_list[start:stop].iterrows():
        print('Current index: {}'.format(index), end='\r')

        appid = row['appid']
        name = row['name']

        # retrive app data for a row, handled by supplied parser, and append to list
        data = parser(appid, name)
        app_data.append(data)

        time.sleep(pause) # prevent overloading api with requests

    return app_data


def process_batches(parser, app_list, download_path, data_filename, index_filename,
                    columns, begin=0, end=-1, batchsize=100, pause=1):
    """Process app data in batches, writing directly to file.

    parser : custom function to format request
    app_list : dataframe of appid and name
    download_path : path to store data
    data_filename : filename to save app data
    index_filename : filename to store highest index written
    columns : column names for file

    Keyword arguments:

    begin : starting index (get from index_filename, default 0)
    end : index to finish (defaults to end of app_list)
    batchsize : number of apps to write in each batch (default 100)
    pause : time to wait after each api request (defualt 1)

    returns: none
    """
    print('Starting at index {}:\n'.format(begin))

    # by default, process all apps in app_list
    if end == -1:
        end = len(app_list) + 1

    # generate array of batch begin and end points
    batches = np.arange(begin, end, batchsize)
    batches = np.append(batches, end)

    apps_written = 0
    batch_times = []

    for i in range(len(batches) - 1):
        start_time = time.time()

        start = batches[i]
        stop = batches[i+1]

        app_data = get_app_data(start, stop, parser, pause)

        rel_path = os.path.join(download_path, data_filename)

        # writing app data to file
        with open(rel_path, 'a', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=columns, extrasaction='ignore')

            for j in range(3,0,-1):
                print("\rAbout to write data, don't stop script! ({})".format(j), end='')
                time.sleep(0.5)

            writer.writerows(app_data)
            print('\rExported lines {}-{} to {}.'.format(start, stop-1, data_filename), end=' ')

        apps_written += len(app_data)

        idx_path = os.path.join(download_path, index_filename)

        # writing last index to file
        with open(idx_path, 'w') as f:
            index = stop
            print(index, file=f)

        # logging time taken
        end_time = time.time()
        time_taken = end_time - start_time

        batch_times.append(time_taken)
        mean_time = statistics.mean(batch_times)

        est_remaining = (len(batches) - i - 2) * mean_time

        remaining_td = date.timedelta(seconds=round(est_remaining))
        time_td = date.timedelta(seconds=round(time_taken))
        mean_td = date.timedelta(seconds=round(mean_time))

        print('Batch {} time: {} (avg: {}, remaining: {})'.format(i, time_td, mean_td, remaining_td))

    print('\nProcessing batches complete. {} apps written'.format(apps_written))

In [ ]:
def get_request(url, parameters=None):
    """Return json-formatted response of a get request using optional parameters.

    Parameters
    ----------
    url : string
    parameters : {'parameter': 'value'}
        parameters to pass as part of get request

    Returns
    -------
    json_data
        json-formatted response (dict-like)
    """
    try:
        response = requests.get(url=url, params=parameters)
    except SSLError as s:
        print('SSL Error:', s)

        for i in range(5, 0, -1):
            print('\rWaiting... ({})'.format(i), end='')
            time.sleep(1)
        print('\rRetrying.' + ' '*10)

        # recusively try again
        return get_request(url, parameters)

    if response:
        return response.json()
    else:
        # response is none usually means too many requests. Wait and try again
        print('No response, waiting 10 seconds...')
        time.sleep(10)
        print('Retrying.')
        return get_request(url, parameters)

In [ ]:
verify = pd.read_csv('/content/drive/My Drive/id_list2.csv')
print(verify.head())
print(verify.shape)

    appid                          name
0  893950             RUSSI.A SIMULATOR
1  893960  Running Naked Simulator 2019
2  894000       Night of the Blood Moon
3  894010          Battle Of Worldviews
4  894020                  Death's Door
(25000, 2)


In [ ]:
app_list = verify

In [ ]:
len(app_list)

25000

In [ ]:
def parse_steamspy_request(appid, name):
    """Parser to handle SteamSpy API data."""
    url = "https://steamspy.com/api.php"
    parameters = {"request": "appdetails", "appid": appid}

    json_data = get_request(url, parameters)
    return json_data


# set files and columns
download_path = '../data/download'
steamspy_data = 'steamspy_data.csv'
steamspy_index = 'steamspy_index.txt'

steamspy_columns = [
    'appid', 'name', 'developer', 'publisher', 'score_rank', 'positive',
    'negative', 'userscore', 'owners', 'average_forever', 'average_2weeks',
    'median_forever', 'median_2weeks', 'price', 'initialprice', 'discount',
    'languages', 'genre', 'ccu', 'tags'
]

reset_index(download_path, steamspy_index)
index = get_index(download_path, steamspy_index)

# Wipe data file if index is 0
prepare_data_file(download_path, steamspy_data, index, steamspy_columns)

process_batches(
    parser=parse_steamspy_request,
    app_list=app_list,
    download_path=download_path,
    data_filename=steamspy_data,
    index_filename=steamspy_index,
    columns=steamspy_columns,
    begin=index,
    end=len(app_list),
    batchsize=5,
    pause=0.3
)

Streaming output truncated to the last 5000 lines.
Exported lines 20-24 to steamspy_data.csv. Batch 4 time: 0:00:04 (avg: 0:00:04, remaining: 5:41:57)
Exported lines 25-29 to steamspy_data.csv. Batch 5 time: 0:00:04 (avg: 0:00:04, remaining: 5:40:39)
Exported lines 30-34 to steamspy_data.csv. Batch 6 time: 0:00:04 (avg: 0:00:04, remaining: 5:40:00)
Exported lines 35-39 to steamspy_data.csv. Batch 7 time: 0:00:04 (avg: 0:00:04, remaining: 5:39:13)
Exported lines 40-44 to steamspy_data.csv. Batch 8 time: 0:00:04 (avg: 0:00:04, remaining: 5:38:44)
Exported lines 45-49 to steamspy_data.csv. Batch 9 time: 0:00:04 (avg: 0:00:04, remaining: 5:38:27)
Exported lines 50-54 to steamspy_data.csv. Batch 10 time: 0:00:04 (avg: 0:00:04, remaining: 5:38:11)
Exported lines 55-59 to steamspy_data.csv. Batch 11 time: 0:00:04 (avg: 0:00:04, remaining: 5:37:50)
Exported lines 60-64 to steamspy_data.csv. Batch 12 time: 0:00:04 (avg: 0:00:04, remaining: 5:37:36)
Exported lines 65-69 to steamspy_data.csv. Bat

In [ ]:
pd.read_csv('../data/download/steamspy_data.csv').head()

,appid,name,developer,publisher,score_rank,positive,negative,userscore,owners,average_forever,average_2weeks,median_forever,median_2weeks,price,initialprice,discount,languages,genre,ccu,tags
0,893950,RUSSI.A SIMULATOR,"Slav Squat Games, CyberPutin2076 Team",Kavkaz Sila Games,NaN,261,97,0,"0 .. 20,000",33,0,47,0,299.0,299.0,0.0,Russian,"Action, Adventure, Simulation",0,"{'Nudity': 35, 'Sexual Content': 33, 'Simulati..."
1,893960,Running Naked Simulator 2019,Shahzeb A.,Kyle K.,NaN,7,4,0,"0 .. 20,000",0,0,0,0,99.0,99.0,0.0,English,"Action, Casual, Indie, Simulation",0,"{'Action': 21, 'Indie': 21, 'Casual': 21, 'Sim..."
2,894000,Night of the Blood Moon,Tyler McDermott,Tyler McDermott,NaN,60,6,0,"0 .. 20,000",0,0,0,0,499.0,499.0,0.0,English,"Action, Adventure, RPG",0,"{'Action': 33, 'Adventure': 30, 'RPG': 30, 'Ac..."
3,894010,Battle Of Worldviews,ARGames,Metal Fox,NaN,10,15,0,"0 .. 20,000",0,0,0,0,75.0,99.0,24.0,English,"Indie, Strategy",0,"{'Strategy': 22, 'Indie': 20}"
4,894020,Death's Door,Acid Nerve,Devolver Digital,NaN,15861,1112,0,"500,000 .. 1,000,000",765,0,754,0,1999.0,1999.0,0.0,"English, French, German, Spanish - Spain, Japa...","Action, Adventure, Indie, RPG",66,"{'Cute': 390, 'Souls-like': 388, '3D Platforme..."


In [ ]:
steamspy_data.csv.to_csv('/content/drive/My Drive/steamspy_data_list_222.csv', index=False)

AttributeError: 'str' object has no attribute 'csv'